# Short-Term & Long-Term Memory

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) LangChain roadmap.

You will learn how to add short-term conversation memory and long-term persistent memory to agents.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Short-Term Memory with InMemorySaver

Use `InMemorySaver` as a checkpointer to persist conversation history across turns.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

model = init_chat_model("gpt-4o-mini", model_provider="openai")
checkpointer = InMemorySaver()

agent = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful assistant. Remember what the user tells you.",
    checkpointer=checkpointer,
)

## 4. Thread IDs for Conversation Persistence

Messages with the same `thread_id` share conversation context.

In [ ]:
config = {"configurable": {"thread_id": "session-1"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="My name is Alice and I'm a software engineer.")]},
    config=config,
)
print("Bot:", result["messages"][-1].content)

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="What's my name and profession?")]},
    config=config,
)
print("Bot:", result["messages"][-1].content)

## 5. Separate Threads Have No Shared Memory

In [ ]:
config_2 = {"configurable": {"thread_id": "session-2"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Do you know my name?")]},
    config=config_2,
)
print("Bot:", result["messages"][-1].content)

## 6. Long-Term Memory with InMemoryStore

Use `InMemoryStore` with namespaces and keys to persist facts across conversations.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

store.put(
    namespace=("users", "alice"),
    key="preferences",
    value={"favorite_color": "blue", "language": "Python", "timezone": "PST"},
)

store.put(
    namespace=("users", "alice"),
    key="bio",
    value={"role": "engineer", "company": "Acme Corp", "years_exp": 5},
)

## 7. store.get and store.search

In [ ]:
item = store.get(namespace=("users", "alice"), key="preferences")
print("Get by key:", item.value)

print("\nSearch namespace:")
items = store.search(namespace=("users", "alice"))
for item in items:
    print(f"  {item.key}: {item.value}")

## 8. Semantic Search with IndexConfig

Enable embedding-based search to find memories by meaning.

In [ ]:
store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
    }
)

store.put(namespace=("knowledge",), key="python-tip", value={"text": "Use list comprehensions for concise loops"})
store.put(namespace=("knowledge",), key="testing-tip", value={"text": "Write unit tests before refactoring"})
store.put(namespace=("knowledge",), key="git-tip", value={"text": "Commit often with meaningful messages"})

results = store.search(namespace=("knowledge",), query="how to write better Python code")
for r in results:
    print(f"{r.key}: {r.value['text']} (score: {r.score:.3f})")

## 9. Combining Both Memory Types

Use short-term memory for conversation context and long-term memory for persistent facts.

In [ ]:
checkpointer = InMemorySaver()
store = InMemoryStore()

store.put(
    namespace=("users", "alice"),
    key="preferences",
    value={"favorite_language": "Python", "experience": "senior"},
)

agent = create_react_agent(
    model=model,
    tools=[],
    prompt="You are a helpful coding assistant.",
    checkpointer=checkpointer,
    store=store,
)

config = {"configurable": {"thread_id": "coding-help"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="Can you suggest a project for me?")]},
    config=config,
)
print("Bot:", result["messages"][-1].content)

## Summary

- Short-term memory uses `InMemorySaver` to persist conversation history within a thread
- `thread_id` groups messages — different IDs create isolated conversations
- Long-term memory uses `InMemoryStore` with namespaces and keys
- `store.put()` saves, `store.get()` retrieves, `store.search()` lists
- Semantic search with `IndexConfig` finds memories by meaning
- Combine both for conversation context + persistent user preferences